In [89]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist 

### Try to model a single agent scheduling and routing problem

Parameters

In [90]:
# constants
INF = 1e2

# number of agents
N = 1

# number of waypoints
M = 2

# number of stages
K = M+1

# generate a dummy mask matrix
mask = np.random.uniform(0,1,(M,M))
mask = mask - np.diag(np.diag(mask))
mask = mask/np.sum(mask, axis=1,keepdims=True)
mask[mask<=1/(M)] = 0.0
mask[mask>1/(M)] = 1.0

# generate a dummy distance matrix between the waypoints
Y = np.random.uniform(0,1,(M,2))
distMat = cdist(Y, Y, 'euclidean')

# agent start and end parameters
start_time = 0.0
s = 0 #int(np.random.choice(range(M)))
d = 1 #int(np.random.choice(range(M)))
assert(s!=d)

# optimization parameters
c0, c1, c2 = 0.1, 10, 1
net_mask = mask.copy()
net_mask[d,:] = 0
net_mask[d,d] = 1
distMat[net_mask == 0.0] = INF

# initialize stage sizes
stage_size = []
for k in range(K+1):
    if k == 0 or k == K:
        stage_size.append(1)
    else:
        stage_size.append(M)


In [91]:
mask, net_mask, distMat

(array([[0., 1.],
        [1., 0.]]),
 array([[0., 1.],
        [0., 1.]]),
 array([[100.        ,   0.37509442],
        [100.        ,   0.        ]]))

Decision Variables

In [92]:
m = gp.Model("single_agent_MIRS")

# Continuous decision variables
theta = {}
for i in range(M):
    theta[i] = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, ub=1000.0, name=f"T{i}")

inv_speed = m.addVar(vtype=GRB.CONTINUOUS, lb=10, ub=100, name=f"inv_speed")

# Stagewise decision variables, and corresponding constraints
Z = {}
eta = {}
Xi = {}
prod_constr = {}

for k in range(K):
    for i in range(stage_size[k]):
        for j in range(stage_size[k+1]):
            Z[k,i,j] = m.addVar(vtype=GRB.CONTINUOUS, name=f"Z_{k}{i}{j}")
            eta[k,i,j] = m.addVar(vtype=GRB.BINARY, name=f"eta_{k}{i}{j}")
            Xi[k,i,j] = m.addVar(vtype=GRB.CONTINUOUS, name=f"Xi_{k}{i}{j}")

            # linearization constraint
            m.addQConstr( Z[k,i,j] == eta[k,i,j]*Xi[k,i,j], name=f"prod_constr_{k}{i}{j}")

            # transition cost constraints
            if k == 0:
                if net_mask[s,j] == 0:
                    expr = INF
                else:
                    travel_time = distMat[s, j] * inv_speed
                    expr0 = c0 * (theta[s] - start_time)*(theta[s] - start_time)
                    expr1 = c1 * (theta[j] - theta[s] - travel_time)*(theta[j] - theta[s] - travel_time)
                    expr2 = c2 * travel_time*travel_time
                    expr = expr0 + expr1 + expr2
            elif k > 0 and k < K-1:
                if net_mask[i,j] == 0:
                    expr = INF
                else:
                    travel_time = distMat[i, j] * inv_speed
                    expr1 = c1 * (theta[j] - theta[i] - travel_time)*(theta[j] - theta[i] - travel_time)
                    expr2 = c2 * travel_time*travel_time
                    expr = expr1 + expr2
            elif k == K-1:
                if net_mask[i,d] == 0:
                    expr = INF
                else:
                    travel_time = distMat[i, d] * inv_speed
                    expr1 = c1 * (theta[d] - theta[i] - travel_time)*(theta[d] - theta[i] - travel_time)
                    expr2 = c2 * travel_time*travel_time
                    expr = expr1 + expr2
            
            m.addConstr(Xi[k,i,j] == expr, name=f"cost_constr_{k}{i}{j}")

        # binary association constraints
        m.addConstr(gp.quicksum(eta[k,i,j] for j in range(stage_size[k+1])) == 1, name=f"sum_eta_{k}{i}")

m.update()
m.Params.NonConvex = 2

print("Number of quadratic constraints:", m.NumQConstrs)
print("Number of linear constraints:", m.NumConstrs)


Set parameter NonConvex to value 2
Number of quadratic constraints: 13
Number of linear constraints: 8


Define objective function

In [93]:
def returnD(Z, distMat):
    U = [None]*K
    for k in range(K):
        # final stage to destination
        if k == 0:
            U[K-1-k] = {}
            j = 0
            for i in range(stage_size[K-1-k]):
                U[K-1-k][i] = Z[K-1-k, i, j]
        elif k > 0 and k < K-1:
            U[K-1-k] = {}
            for i in range(stage_size[K-1-k]):
                expr = []
                for j in range(stage_size[K-k]):
                    expr.append((Z[K-1-k,i,j] + U[K-k][j]))
                U[K-1-k][i] = sum(expr)
        elif k == K-1:
            U[K-1-k] = {}
            expr = []
            i = 0
            for j in range(stage_size[K-k]):
                expr.append((Z[K-1-k,i,j] + U[K-k][j]))
            U[K-1-k] = sum(expr)

    return U[0]

In [94]:
obj = returnD(Z, distMat)
m.setObjective(obj, GRB.MINIMIZE)
m.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G419)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
NonConvex  2

Optimize a model with 8 rows, 27 columns and 11 nonzeros
Model fingerprint: 0x7c19940b
Model has 13 quadratic constraints
Variable types: 19 continuous, 8 integer (8 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 2e+01]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [1e+00, 2e+00]
  Bounds range     [1e+00, 1e+03]
  RHS range        [1e+00, 1e+02]
Presolve removed 8 rows and 20 columns
Presolve time: 0.00s
Presolved: 27 rows, 20 columns, 68 nonzeros
Presolved model has 4 SOS constraint(s)
Presolved model has 6 bilinear constraint(s)

Solving non-convex MIQCP

Variable types: 16 continuous, 4 integer (4 binary)

Root relaxation: objective 0.000000e+00, 5 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Cur

Stagewise cost

In [95]:
# print start and goal locations
print(f"\n--- start-goal locations ---")
print(f"Start:{s}\tGoal:{d}")

# Distance matrix
print(f"\n --- Distance matrix --- ")
print(f"{distMat}")

# Print scalar stagewise variables
print("\n--- Continuous variables theta ---")
for i in range(M):
    print(f"theta[{i}] = {theta[i].X}")

print("\n--- Continuous variable inv_speed ---")
print(f"inv_speed = {inv_speed.X}")

print("\n--- Stagewise variables ---")
for k in range(K):
    for i in range(stage_size[k]):
        for j in range(stage_size[k+1]):
            print(f"Z[{k},{i},{j}] = {Z[k,i,j].X:.3e}, \t eta[{k},{i},{j}] = {eta[k,i,j].X},\t Xi[{k},{i},{j}] = {Xi[k,i,j].X:.3e}")




--- start-goal locations ---
Start:0	Goal:1

 --- Distance matrix --- 
[[100.           0.37509442]
 [100.           0.        ]]

--- Continuous variables theta ---
theta[0] = 0.0012095299552655771
theta[1] = 3.752221501443269

--- Continuous variable inv_speed ---
inv_speed = 10.0

--- Stagewise variables ---
Z[0,0,0] = 0.000e+00, 	 eta[0,0,0] = 0.0,	 Xi[0,0,0] = 1.000e+02
Z[0,0,1] = 1.407e+01, 	 eta[0,0,1] = 1.0,	 Xi[0,0,1] = 1.407e+01
Z[1,0,0] = 0.000e+00, 	 eta[1,0,0] = 0.0,	 Xi[1,0,0] = 1.000e+02
Z[1,0,1] = 1.407e+01, 	 eta[1,0,1] = 1.0,	 Xi[1,0,1] = 1.407e+01
Z[1,1,0] = 0.000e+00, 	 eta[1,1,0] = 0.0,	 Xi[1,1,0] = 1.000e+02
Z[1,1,1] = 0.000e+00, 	 eta[1,1,1] = 1.0,	 Xi[1,1,1] = 0.000e+00
Z[2,0,0] = 1.407e+01, 	 eta[2,0,0] = 1.0,	 Xi[2,0,0] = 1.407e+01
Z[2,1,0] = 0.000e+00, 	 eta[2,1,0] = 1.0,	 Xi[2,1,0] = 0.000e+00
